# Práctica 8 · Segmentación de clientes con K-Means
**Trabajo individual · Python y Google Colab**

Objetivo: descubrir perfiles de compra sin proporcionar categorías previas al algoritmo.
Los datos son los 15 clientes ficticios del PDF. No representan clientes reales.

## Cómo ejecutar
En Google Colab, selecciona **Archivo → Subir cuaderno** y carga este archivo. Después selecciona **Entorno de ejecución → Ejecutar todas**. Las bibliotecas utilizadas están disponibles en Colab.


In [ ]:
# 1. Importar las bibliotecas necesarias.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from IPython.display import display

# 2. Cargar exactamente los datos indicados en el PDF.
datos = {
    "Cliente": [f"C{i:02d}" for i in range(1, 16)],
    "Gasto_mensual": [500,700,600,900,800,4500,5000,4800,5500,4700,10000,11000,9500,12000,10500],
    "Numero_compras": [1,2,1,2,3,8,9,7,10,8,15,17,14,18,16]
}
df = pd.DataFrame(datos)
print("DATOS ORIGINALES: 15 CLIENTES")
display(df)


## Preparación y entrenamiento
El identificador del cliente no participa en el entrenamiento. `StandardScaler` estandariza cada variable: resta su media y divide entre su desviación estándar. Así, los pesos y el número de compras tienen escalas comparables.

`n_clusters=3` fija la cantidad de grupos solicitada; el algoritmo descubre qué clientes pertenecen a cada grupo. `random_state=42` y `n_init=10` hacen reproducible la ejecución en un mismo entorno.


In [ ]:
# 3. Seleccionar únicamente las variables numéricas.
X = df[["Gasto_mensual", "Numero_compras"]]
# 4. Estandarizar antes de calcular las distancias.
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)
# 5. Entrenar K-Means y asignar los grupos.
modelo = KMeans(n_clusters=3, random_state=42, n_init=10)
df["Cluster"] = modelo.fit_predict(X_escalado)
# 6. Resumir y poner nombres DESPUÉS del entrenamiento.
resumen = df.groupby("Cluster").agg(
    Clientes=("Cliente", "count"),
    Gasto_promedio=("Gasto_mensual", "mean"),
    Compras_promedio=("Numero_compras", "mean")
)
orden = resumen.sort_values("Gasto_promedio").index
perfiles = dict(zip(orden, ["Compra ocasional", "Compra frecuente", "Alto consumo"]))
df["Perfil"] = df["Cluster"].map(perfiles)
resumen["Perfil"] = resumen.index.map(perfiles)
print("CLIENTES AGRUPADOS")
display(df)
print("RESUMEN DE LOS TRES GRUPOS")
display(resumen.sort_values("Gasto_promedio").round(2))


In [ ]:
# 7. Graficar en unidades originales para facilitar la interpretación.
colores = ["#2563eb", "#d97706", "#059669"]
centros = escalador.inverse_transform(modelo.cluster_centers_)
fig, ax = plt.subplots(figsize=(11, 6))
for color, cluster in zip(colores, orden):
    grupo = df[df["Cluster"] == cluster]
    ax.scatter(grupo["Gasto_mensual"], grupo["Numero_compras"],
               color=color, s=85, label=perfiles[cluster])
    for _, cliente in grupo.iterrows():
        ax.annotate(cliente["Cliente"], (cliente["Gasto_mensual"], cliente["Numero_compras"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.scatter(centros[:,0], centros[:,1], marker="X", color="black", s=150, label="Centroides")
ax.set(xlabel="Gasto mensual (MXN)", ylabel="Número de compras al mes",
       title="Segmentación de 15 clientes con K-Means · 3 grupos")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig("clusters_3.png", dpi=160)
plt.show()


## Experimentación: 2, 3 y 4 clusters
Se mantienen los mismos datos, escalas y parámetros. La silueta mide separación y cohesión: valores cercanos a 1 indican grupos mejor separados. Es una ayuda para comparar este conjunto, no una prueba de que exista una clasificación verdadera.

In [ ]:
resultados = []
resumenes = {}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
for ax, k in zip(axes, [2, 3, 4]):
    experimento = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = experimento.fit_predict(X_escalado)
    temporal = df[["Cliente", "Gasto_mensual", "Numero_compras"]].copy()
    temporal["Cluster"] = etiquetas
    tabla = temporal.groupby("Cluster").agg(
        Clientes=("Cliente", "count"),
        Integrantes=("Cliente", lambda s: ", ".join(s)),
        Gasto_promedio=("Gasto_mensual", "mean"),
        Compras_promedio=("Numero_compras", "mean"))
    resumenes[k] = tabla.sort_values("Gasto_promedio")
    resultados.append({"Clusters": k, "Silueta": silhouette_score(X_escalado, etiquetas),
                       "Inercia": experimento.inertia_})
    for j, cluster in enumerate(resumenes[k].index):
        grupo = temporal[temporal["Cluster"] == cluster]
        ax.scatter(grupo["Gasto_mensual"], grupo["Numero_compras"], s=65,
                   color=["#2563eb", "#d97706", "#059669", "#9333ea"][j],
                   label=f"Grupo {j+1}")
    ax.set(title=f"{k} clusters", xlabel="Gasto mensual (MXN)")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Compras al mes")
fig.suptitle("Comparación de agrupaciones sobre los mismos datos")
fig.tight_layout()
fig.savefig("comparacion_clusters.png", dpi=160)
plt.show()
comparacion = pd.DataFrame(resultados)
display(comparacion.round(4))
for k, tabla in resumenes.items():
    print(f"INTEGRANTES Y PROMEDIOS CON {k} CLUSTERS")
    display(tabla.round(2))
# Guardar resultados que también se pueden descargar desde Archivos de Colab.
df.to_csv("clientes_segmentados.csv", index=False)
resumen.to_csv("resumen_clusters.csv")
comparacion.to_csv("comparacion_clusters.csv", index=False)
